In [167]:
import findspark
findspark.init()

In [168]:
from pyspark.sql import SparkSession
from pathlib import Path

spark = SparkSession.builder.appName("Flights").getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", "5")

In [169]:
current_dir = Path.cwd().parent
directory_path = current_dir / "data" / "retail-data" / "by-day"
data_path = str(directory_path) + "/*csv"
staticDataFrame = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(data_path)

In [170]:
staticDataFrame.createOrReplaceTempView("retail_data")
staticSchema = staticDataFrame.schema

In [171]:
from pyspark.sql.functions import window, column, desc, col

staticDataFrame \
    .selectExpr(
        "CustomerId",
        "(UnitPrice * Quantity) as total_cost",
        "InvoiceDate") \
    .groupBy(
        col("CustomerId"), window(col("InvoiceDate"), "1 day"))\
    .sum("total_cost") \
    .show()





+----------+--------------------+------------------+
|CustomerId|              window|   sum(total_cost)|
+----------+--------------------+------------------+
|   14075.0|{2011-12-05 01:00...|316.78000000000003|
|   18180.0|{2011-12-05 01:00...|            310.73|
|   15358.0|{2011-12-05 01:00...| 830.0600000000003|
|   15392.0|{2011-12-05 01:00...|304.40999999999997|
|   15290.0|{2011-12-05 01:00...|263.02000000000004|
|   16811.0|{2011-12-05 01:00...|             232.3|
|   12748.0|{2011-12-05 01:00...| 363.7899999999999|
|   16500.0|{2011-12-05 01:00...| 52.74000000000001|
|   16873.0|{2011-12-05 01:00...|1854.8300000000002|
|   14060.0|{2011-12-05 01:00...|297.47999999999996|
|   14649.0|{2011-12-05 01:00...| 513.9899999999998|
|   16904.0|{2011-12-05 01:00...| 349.0200000000001|
|   17857.0|{2011-12-05 01:00...|            2979.6|
|   14083.0|{2011-12-05 01:00...| 446.5700000000001|
|   14777.0|{2011-12-05 01:00...|             -2.95|
|   16684.0|{2011-12-05 01:00...| 5401.9799999

In [172]:
streamingDataFrame = spark.readStream \
    .schema(staticSchema) \
    .option("maxFilesPerTrigger", 1) \
    .format("csv") \
    .option("header", "true") \
    .load(data_path)

In [173]:
streamingDataFrame.isStreaming

True

In [174]:
purchaseByCustomerPerHour = streamingDataFrame \
    .selectExpr(
        "CustomerId",
        "(UnitPrice * Quantity) as total_cost",
        "InvoiceDate") \
    .groupBy(
        col("CustomerId"), window(col("InvoiceDate"), "1 day")) \
    .sum("total_cost")

In [195]:
streaming_data = purchaseByCustomerPerHour.writeStream \
    .format("memory") \
    .queryName("customer_purchases") \
    .outputMode("complete") \
    .start()


In [202]:
streaming_data.stop()

In [208]:
spark.sql("""
    SELECT * 
    FROM customer_purchases
    ORDER BY `sum(total_cost)` DESC
    """) \
    .show()

+----------+--------------------+------------------+
|CustomerId|              window|   sum(total_cost)|
+----------+--------------------+------------------+
|      null|{2010-12-03 01:00...| 23021.99999999999|
|      null|{2010-12-01 01:00...|12584.299999999988|
|   15061.0|{2010-12-02 01:00...| 9407.339999999998|
|   13777.0|{2010-12-01 01:00...|           6585.16|
|   17850.0|{2010-12-02 01:00...|3891.8699999999985|
|   16029.0|{2010-12-01 01:00...|           3702.12|
|   16210.0|{2010-12-01 01:00...|2474.7399999999993|
|   13081.0|{2010-12-03 01:00...|           2366.78|
|   16754.0|{2010-12-02 01:00...|            2002.4|
|   12433.0|{2010-12-01 01:00...|1919.1400000000008|
|   15299.0|{2010-12-02 01:00...|1835.0100000000002|
|   17511.0|{2010-12-01 01:00...|           1825.74|
|   14031.0|{2010-12-02 01:00...|1714.2500000000005|
|   14911.0|{2010-12-03 01:00...|1705.6499999999994|
|   14680.0|{2010-12-03 01:00...|1680.8799999999999|
|   13013.0|{2010-12-03 01:00...|1528.34000000

In [209]:
spark.stop()